# RECAST

Small project to swap a character's face in a video clip with a reference photo, without touching anything else in the scene (other people, background, dialogue timing).

Runs on Kaggle's free GPU. Upload a video and a few photos of the person you want to insert, run the cells top to bottom, download the result at the end.

Turn on GPU (Settings > Accelerator) and Internet (Settings > Internet) before running.

### Setup

In [ ]:
!pip install -q insightface onnxruntime-gpu opencv-python-headless gfpgan basicsr facexlib ipywidgets!apt-get -qq install -y ffmpeg

Grabbing the two models this needs — the swap model and the face-restoration model.

In [ ]:
import osos.makedirs('models', exist_ok=True)if not os.path.exists('models/inswapper_128.onnx'):    !wget -q -O models/inswapper_128.onnx https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnxif not os.path.exists('GFPGANv1.4.pth'):    !wget -q https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pthprint('Models ready.')

### Upload

Video first, then a few photos of the face you want to use instead — 2 to 5 works well, mix in a side angle if you have one. Sharper photos give a noticeably better result than blurry ones.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

video_upload = widgets.FileUpload(accept='video/*', multiple=False, description='Upload Video')
photo_upload = widgets.FileUpload(accept='image/*', multiple=True, description='Upload Photos')

print('Select your video (single file) and 2-5 photos (multi-select) below. Wait for filenames to appear, then run the next cell.')
display(video_upload, photo_upload)

In [ ]:
import os

os.makedirs('/kaggle/working/uploads', exist_ok=True)

def _save_single(uploader, label):
    if not uploader.value:
        raise ValueError(f'No file uploaded for {label} yet — select a file above, then re-run this cell.')
    value = uploader.value
    if isinstance(value, (tuple, list)):
        item = value[0]
        name, content = item['name'], item['content']
    else:
        name = list(value.keys())[0]
        content = value[name]['content']
    dest = f'/kaggle/working/uploads/{name}'
    with open(dest, 'wb') as f:
        f.write(bytes(content))
    return dest

def _save_multiple(uploader, label):
    if not uploader.value:
        raise ValueError(f'No files uploaded for {label} yet — select files above, then re-run this cell.')
    value = uploader.value
    paths = []
    if isinstance(value, (tuple, list)):
        items = [{'name': it['name'], 'content': it['content']} for it in value]
    else:
        items = [{'name': k, 'content': v['content']} for k, v in value.items()]
    for item in items:
        dest = f"/kaggle/working/uploads/{item['name']}"
        with open(dest, 'wb') as f:
            f.write(bytes(item['content']))
        paths.append(dest)
    return paths

video_path = _save_single(video_upload, 'video')
photo_paths = _save_multiple(photo_upload, 'photos')

print('Video saved to:', video_path)
print(f'{len(photo_paths)} photo(s) saved:')
for p in photo_paths:
    print(' -', p)

Loading the face detector and the swap model onto the GPU.

In [ ]:
import cv2import numpy as npimport insightfacefrom insightface.app import FaceAnalysisface_analyzer = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])face_analyzer.prepare(ctx_id=0, det_size=(640, 640))swapper = insightface.model_zoo.get_model('models/inswapper_128.onnx', providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])print('Models loaded.')

Averaging the face across however many photos got uploaded, instead of relying on just one. Single photos can be biased by lighting or angle, so this smooths that out a bit.

In [ ]:
class AveragedSourceFace:
    """Minimal stand-in for insightface's Face object — the swapper only needs
    .normed_embedding (identity vector), and our own cosine_sim() only needs .embedding."""
    def __init__(self, embedding):
        self.embedding = embedding
        norm = np.linalg.norm(embedding) + 1e-6
        self.normed_embedding = embedding / norm

embeddings = []
used_photos = []

for path in photo_paths:
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    if sharpness < 60:
        print(f'Warning: {os.path.basename(path)} looks soft/blurry (score {sharpness:.1f}) — still using it, but a sharper photo would help.')

    faces = face_analyzer.get(img)
    if len(faces) == 0:
        print(f'No face detected in {os.path.basename(path)} — skipping this photo.')
        continue

    largest = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))
    embeddings.append(largest.embedding)
    used_photos.append(path)

if len(embeddings) == 0:
    raise ValueError('No face detected in any of the uploaded photos. Try clearer, front-facing shots.')

avg_embedding = np.mean(embeddings, axis=0)
source_face = AveragedSourceFace(avg_embedding)

print(f'Reference identity built from {len(used_photos)} photo(s): {[os.path.basename(p) for p in used_photos]}')

Before swapping anything, it needs to know who to swap. It checks the first ~60 frames for a face and remembers it — that becomes "the character." Everyone else in the video is left alone.

In [ ]:
def cosine_sim(a, b):    a = a / (np.linalg.norm(a) + 1e-6)    b = b / (np.linalg.norm(b) + 1e-6)    return float(np.dot(a, b))IDENTITY_THRESHOLD = 0.18  # lowered a lot — prioritizes catching every frame with the main character over strict precisioncap = cv2.VideoCapture(video_path)target_identity_embedding = Nonechecked = 0while checked < 60:    ret, frame = cap.read()    if not ret:        break    faces = face_analyzer.get(frame)    if len(faces) > 0:        largest = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]))        target_identity_embedding = largest.embedding        break    checked += 1cap.release()if target_identity_embedding is None:    raise ValueError('Could not find any face in the first 60 frames. Trim the clip so the main character appears near the start.')print(f'Main character locked from frame {checked}.')

The new face and the original scene were shot in different lighting, so pasting one onto the other looks off unless the colors are nudged to match. This blends 70% toward the scene's lighting and keeps 30% of the photo's own tone, so it doesn't lose the person's actual skin tone completely.

In [ ]:
COLOR_MATCH_STRENGTH = 0.7def match_color(swapped_region, original_region, strength=COLOR_MATCH_STRENGTH):    swapped_lab = cv2.cvtColor(swapped_region, cv2.COLOR_BGR2LAB).astype(np.float32)    original_lab = cv2.cvtColor(original_region, cv2.COLOR_BGR2LAB).astype(np.float32)    for c in range(3):        s_mean, s_std = swapped_lab[:, :, c].mean(), swapped_lab[:, :, c].std() + 1e-6        o_mean, o_std = original_lab[:, :, c].mean(), original_lab[:, :, c].std() + 1e-6        target_mean = s_mean + (o_mean - s_mean) * strength        target_std = s_std + (o_std - s_std) * strength        swapped_lab[:, :, c] = (swapped_lab[:, :, c] - s_mean) * (target_std / s_std) + target_mean    return cv2.cvtColor(np.clip(swapped_lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2BGR)

Main loop. Every frame gets checked against the locked identity from before, and whichever face matches gets swapped. Thresholds are set loose here on purpose — earlier versions skipped too many frames trying to be careful about odd angles, which just meant chunks of the video weren't swapped at all.

In [ ]:
from tqdm import tqdmdef estimate_yaw(kps):    """Rough left-right turn estimate from 5-point kps, using how far the nose sits    from the midpoint between the eyes, relative to the eye distance.    ~0 = facing camera, closer to +/-1 = turned to the side.    """    left_eye, right_eye, nose = kps[0], kps[1], kps[2]    eye_center_x = (left_eye[0] + right_eye[0]) / 2    eye_dist = abs(right_eye[0] - left_eye[0]) + 1e-6    return (nose[0] - eye_center_x) / (eye_dist / 2)YAW_SKIP_THRESHOLD = 0.95   # relaxed a lot — only near-total profile (almost fully sideways) gets skipped nowDET_SCORE_MIN = 0.20        # relaxed — only very low-confidence detections get skipped nowcap = cv2.VideoCapture(video_path)fps = cap.get(cv2.CAP_PROP_FPS)width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))out = cv2.VideoWriter('swapped_silent.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))no_match = 0skipped_angle = 0pbar = tqdm(total=total_frames)while True:    ret, frame = cap.read()    if not ret:        break    faces = face_analyzer.get(frame)    target_face, best_sim = None, -1    for f in faces:        sim = cosine_sim(f.embedding, target_identity_embedding)        if sim > best_sim:            best_sim, target_face = sim, f    if target_face is None or best_sim < IDENTITY_THRESHOLD:        no_match += 1        out.write(frame)        pbar.update(1)        continue    yaw = estimate_yaw(target_face.kps)    if abs(yaw) > YAW_SKIP_THRESHOLD or target_face.det_score < DET_SCORE_MIN:        skipped_angle += 1        out.write(frame)  # too extreme an angle to swap convincingly — keep the original face here        pbar.update(1)        continue    original_frame = frame.copy()    swapped = swapper.get(frame, target_face, source_face, paste_back=True)    x1, y1, x2, y2 = map(int, target_face.bbox)    x1, y1 = max(0, x1), max(0, y1)    x2, y2 = min(width, x2), min(height, y2)    if x2 > x1 and y2 > y1:        matched = match_color(swapped[y1:y2, x1:x2], original_frame[y1:y2, x1:x2])        swapped[y1:y2, x1:x2] = matched    frame = swapped    out.write(frame)    pbar.update(1)cap.release()out.release()pbar.close()print(f'Swap pass done. {no_match} frames had no confident identity match, {skipped_angle} frames skipped for extreme angle (both kept as original).')

Small compatibility fix — GFPGAN was written against an older torchvision and breaks on the current one because a function got moved. This patches it back in before importing GFPGAN below.

In [ ]:
import sysimport typesimport torchvision.transforms.functional as Ffake_module = types.ModuleType('torchvision.transforms.functional_tensor')fake_module.rgb_to_grayscale = F.rgb_to_grayscalesys.modules['torchvision.transforms.functional_tensor'] = fake_moduleprint('Patch applied — GFPGAN will import cleanly now.')

GFPGAN sharpens the face but it also has a habit of "fixing" blinks and mouth movement into a generic, static look, which ends up looking worse than the original. So this only lets it touch skin/jaw detail and pastes the original eyes and mouth back in from before restoration.

In [ ]:
from gfpgan import GFPGANerrestorer = GFPGANer(model_path='GFPGANv1.4.pth', upscale=1, arch='clean', channel_multiplier=2)def feature_mask_from_points(points, shape, radius):    h, w = shape[:2]    mask = np.zeros((h, w), dtype=np.float32)    for (px, py) in points:        cv2.circle(mask, (int(px), int(py)), int(radius), 1.0, -1)    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=max(3, radius * 0.5))    if mask.max() > 0:        mask = mask / mask.max()    return maskdef eyes_and_mouth_mask(kps, shape):    left_eye, right_eye, nose, left_mouth, right_mouth = kps    eye_dist = np.linalg.norm(np.array(left_eye) - np.array(right_eye))    eye_mask = feature_mask_from_points([left_eye, right_eye], shape, radius=max(6, eye_dist * 0.3))    mouth_center = ((left_mouth[0] + right_mouth[0]) / 2, (left_mouth[1] + right_mouth[1]) / 2)    mouth_width = np.linalg.norm(np.array(left_mouth) - np.array(right_mouth))    mouth_mask = feature_mask_from_points([mouth_center], shape, radius=max(10, mouth_width * 0.9))    return np.clip(eye_mask + mouth_mask, 0, 1)[:, :, None]cap = cv2.VideoCapture('swapped_silent.mp4')fps = cap.get(cv2.CAP_PROP_FPS)width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))out = cv2.VideoWriter('swapped_restored_silent.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))pbar = tqdm(total=total_frames)MARGIN = 0.35RESTORE_WEIGHT = 0.2RESTORE_IDENTITY_THRESHOLD = 0.18  # matches the relaxed threshold used in the swap passwhile True:    ret, frame = cap.read()    if not ret:        break    faces = face_analyzer.get(frame)    target_face, best_sim = None, -1    for f in faces:        sim = cosine_sim(f.embedding, source_face.embedding)  # matching the SOURCE identity now, since it's already swapped in        if sim > best_sim:            best_sim, target_face = sim, f    if target_face is None or best_sim < RESTORE_IDENTITY_THRESHOLD:        out.write(frame)  # main character not found this frame — leave as-is, don't touch other people        pbar.update(1)        continue    x1, y1, x2, y2 = target_face.bbox    bw, bh = x2 - x1, y2 - y1    x1 = max(0, int(x1 - bw * MARGIN)); y1 = max(0, int(y1 - bh * MARGIN))    x2 = min(width, int(x2 + bw * MARGIN)); y2 = min(height, int(y2 + bh * MARGIN))    if x2 <= x1 or y2 <= y1:        out.write(frame)        pbar.update(1)        continue    original_crop = frame[y1:y2, x1:x2].copy()    _, _, restored_crop = restorer.enhance(        original_crop, has_aligned=False, only_center_face=True, paste_back=True, weight=RESTORE_WEIGHT    )    if restored_crop.shape[:2] != original_crop.shape[:2]:        restored_crop = cv2.resize(restored_crop, (original_crop.shape[1], original_crop.shape[0]))    kps_local = [(kx - x1, ky - y1) for kx, ky in target_face.kps]    preserve_mask = eyes_and_mouth_mask(kps_local, original_crop.shape)    blended_crop = (restored_crop.astype(np.float32) * (1 - preserve_mask) +                    original_crop.astype(np.float32) * preserve_mask).astype(np.uint8)    frame[y1:y2, x1:x2] = blended_crop    out.write(frame)    pbar.update(1)cap.release()out.release()pbar.close()print('Restoration pass done (identity-locked, eyes + mouth preserved, light touch).')

This step sharpens the whole frame further, but it also tends to make the video look processed/artificial. Off by default — flip `APPLY_UPSCALE` to `True` if a sharper export matters more than a natural look.

In [ ]:
APPLY_UPSCALE = Falseif APPLY_UPSCALE:    if not os.path.exists('RealESRGAN_x2plus.pth'):        !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth    from basicsr.archs.rrdbnet_arch import RRDBNet    from realesrgan import RealESRGANer    upsampler_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=2)    upsampler = RealESRGANer(scale=2, model_path='RealESRGAN_x2plus.pth', model=upsampler_model, tile=400, half=True)    cap = cv2.VideoCapture('swapped_restored_silent.mp4')    fps = cap.get(cv2.CAP_PROP_FPS)    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))    ret, frame = cap.read()    h, w = frame.shape[:2]    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)    out = cv2.VideoWriter('swapped_sharp_silent.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w*2, h*2))    pbar = tqdm(total=total_frames)    while True:        ret, frame = cap.read()        if not ret:            break        sharpened, _ = upsampler.enhance(frame, outscale=2)        out.write(sharpened)        pbar.update(1)    cap.release()    out.release()    pbar.close()    print('Upscale pass done.')else:    print('Upscale skipped — using Step 10 output directly for a natural look.')

Original audio never got touched, so it just gets muxed back onto the final video here.

In [ ]:
silent_source = 'swapped_sharp_silent.mp4' if os.path.exists('swapped_sharp_silent.mp4') else 'swapped_restored_silent.mp4'final_output = 'recast_final_v6.mp4'!ffmpeg -y -i "{silent_source}" -i "{video_path}" -c:v libx264 -crf 18 -map 0:v:0 -map 1:a:0? -shortest "{final_output}"print('Final video ready:', final_output)

In [ ]:
print('Done. File ready at:', os.path.abspath('recast_final_v6.mp4'))print('Download it from the Output panel on the right side of this notebook.')

In [ ]:
from IPython.display import FileLink
FileLink('recast_final_v6.mp4')

---

A couple of things worth knowing if you're poking at this:

The swap model struggles with side profiles — it just wasn't trained on much of that, so very sideways frames can look a bit rough. Loosening `IDENTITY_THRESHOLD` gets more frames swapped at the cost of some of those looking rougher; tightening it back up trades coverage for cleaner frames. There's no fixing this outright without a different underlying model.

Also worth remembering this is frame-by-frame, so there's no real temporal smoothing — fast motion can flicker a little between frames.

Use your own footage/photos, or stuff you actually have permission to use.